## SAM 2 ENTRENADO

In [1]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

# TARDA MUCHO PORQUE TIENE QUE COGER TODOS LOS PUNTOS DE LA MÁSCARA
"""def assd(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    
    d_pred_to_gt = cKDTree(gt_points).query(pred_points)[0].mean()
    d_gt_to_pred = cKDTree(pred_points).query(gt_points)[0].mean()
    
    return (d_pred_to_gt + d_gt_to_pred) / 2"""

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [2]:
import time
import torch

def measure_inference(predictor, image, point_coords, point_labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return masks, scores, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2

    start = time.time()
    predictor.set_image(img_path)
    predictor.model.set_classes(text=["polyp"])
    predictor.prompts["text"] = [text_prompt]
    results = predictor()
    
    latency = (time.time() - start) * 1000

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [4]:
import os
import shutil
import random

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\PASCAL-S\\Test"

def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "Image", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "GT",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "Image", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".png"), os.path.join(dataset, "GT",  split, name + ".png"))

split_dataset(dataset)


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import cv2
import numpy as np
import os
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "Image", split)
        self.masks_dir  = os.path.join(dataset_path, "GT",  split)
        self.samples    = []

        for img_name in os.listdir(self.images_dir):
            if not img_name.endswith(".jpg"):
                continue
            name = img_name.replace(".jpg", "")
            img_path  = os.path.join(self.images_dir, img_name)
            mask_path = os.path.join(self.masks_dir,  name + ".png")
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (256, 256))
        mask_bin = (mask > 127).astype(np.float32)

        ys, xs = np.where(mask_bin > 0)
        if len(xs) > 0:
            cx, cy = float(xs.mean()), float(ys.mean())
        else:
            cx, cy = mask_bin.shape[1] / 2, mask_bin.shape[0] / 2

        mask_tensor = torch.tensor(mask_bin).unsqueeze(0)
        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])

        return image, mask_tensor, point, label


def train_sam(dataset_path, weights_path, config_path, output_weights, epochs=50, batch_size=4):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)

    for param in sam2.image_encoder.parameters():
        param.requires_grad = False
    for param in sam2.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam2.sam_mask_decoder.parameters(), lr=1e-4)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam2.train()
    predictor = SAM2ImagePredictor(sam2)

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):
                image_np = (images[i].permute(1, 2, 0).numpy() * 255).astype(np.uint8)

                with torch.no_grad():
                    predictor.set_image(image_np)

                sparse_embeddings, dense_embeddings = sam2.sam_prompt_encoder(
                    points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                    boxes=None,
                    masks=None
                )

                image_embedding = predictor._features["image_embed"]
                image_pe        = sam2.sam_prompt_encoder.get_dense_pe()
                high_res_features = predictor._features["high_res_feats"]

                low_res_masks, _, _, _ = sam2.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    torch.save({"model": sam2.state_dict()}, output_weights)
    return output_weights


## SAM 2 BASE 

In [ ]:
import numpy as np
import os
import cv2
import pandas as pd
import json
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\PASCAL-S\\Test"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
config_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\sam2\\sam2\\configs\\sam2\\sam2_hiera_b+.yaml"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam2b_pascals.pt"


def evaluate_model(dataset, weights_path, config_path):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)
    sam2.eval()
    predictor = SAM2ImagePredictor(sam2)

    images_dir = os.path.join(dataset, "Image", "test")
    masks_dir = os.path.join(dataset, "GT", "test")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_file in os.listdir(images_dir):
        if not img_file.endswith(".jpg"):
            continue
        name = img_file.replace(".jpg", "")
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        ys, xs = np.where(gt_mask)
        if len(xs) > 0:
            cx, cy = float(xs.mean()), float(ys.mean())
        else:
            cx, cy = gt_mask.shape[1] / 2, gt_mask.shape[0] / 2

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        masks, scores, latency, vram = measure_inference(predictor, image, np.array([[cx, cy]]), np.array([1]))

        if masks is None or len(masks) == 0:
            continue
        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_2_b_pascal"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

  
trained_weights = train_sam(dataset, weights, config_path, output_weights)
evaluate_model(dataset, trained_weights, config_path)


## SAM 2 GRANDE

In [ ]:
import numpy as np
import os
import cv2
import pandas as pd
import json
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\PASCAL-S\\Test"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
config_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\sam2\\sam2\\configs\\sam2\\sam2_hiera_l.yaml"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam2l_pascals.pt"


def evaluate_model(dataset, weights_path, config_path):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)
    sam2.eval()
    predictor = SAM2ImagePredictor(sam2)

    images_dir = os.path.join(dataset, "Image", "test")
    masks_dir = os.path.join(dataset, "GT", "test")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_file in os.listdir(images_dir):
        if not img_file.endswith(".jpg"):
            continue
        name = img_file.replace(".jpg", "")
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        ys, xs = np.where(gt_mask)
        if len(xs) > 0:
            cx, cy = float(xs.mean()), float(ys.mean())
        else:
            cx, cy = gt_mask.shape[1] / 2, gt_mask.shape[0] / 2

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        masks, scores, latency, vram = measure_inference(predictor, image, np.array([[cx, cy]]), np.array([1]))

        if masks is None or len(masks) == 0:
            continue
        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_2_l_pascal"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

  
trained_weights = train_sam(dataset, weights, config_path, output_weights)
evaluate_model(dataset, trained_weights, config_path)


MissingConfigException: Cannot find primary config 'C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\pesos_ultralytics\sam2_l.pt'. Check that it's in your config search path.

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=pkg://sam2
	provider=schema, path=structured://